# 05 Models — LightGBM — Women's

LightGBM as a fast alternative to XGBoost. Uses the same LOGO-CV framework with isotonic calibration.

**Inputs** (from S3 `04_preprocessing/womens/`):
- `train_features.parquet`, `stage1_features.parquet`, `stage2_features.parquet`, `feature_columns.parquet`

**Outputs** (to S3 `05_models/lightgbm/womens/`):
- `oof_predictions.parquet`, `stage1_predictions.parquet`, `stage2_predictions.parquet`, `cv_results.parquet`

In [1]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

try:
    import lightgbm as lgb
except:
    !pip install lightgbm
    import lightgbm as lgb

from sklearn.metrics import brier_score_loss
from sklearn.isotonic import IsotonicRegression

print(f"LightGBM version: {lgb.__version__}")

LightGBM version: 4.6.0


#### Functions

In [2]:
def read_parquet(filename):
    """Read parquet from S3 or local."""
    try:
        return pd.read_parquet(f"{INPUT_PREFIX}{filename}")
    except Exception:
        return pd.read_parquet(f"../../04_preprocessing/output/{filename}")

def lgb_brier_objective(preds, train_data):
    """Custom Brier objective for LightGBM. Note: preds are raw (pre-sigmoid)."""
    labels = train_data.get_label()
    p = 1.0 / (1.0 + np.exp(-preds))
    grad = 2.0 * (p - labels) * p * (1.0 - p)
    hess = 2.0 * p * (1.0 - p) * (p * (1.0 - p) + (p - labels) * (1.0 - 2.0 * p))
    hess = np.maximum(hess, 1e-6)
    return grad, hess

def lgb_brier_eval(preds, train_data):
    """Custom Brier evaluation metric for LightGBM."""
    labels = train_data.get_label()
    p = 1.0 / (1.0 + np.exp(-preds))
    brier = np.mean((p - labels) ** 2)
    return 'brier', brier, False  # False = lower is better

#### Constants

In [3]:
BUCKET = "march-machine-learning-mania-2026"
GENDER = "womens"
STAGE = "05_models/lightgbm"
INPUT_PREFIX = f"s3://{BUCKET}/04_preprocessing/{GENDER}/"
OUTPUT_PREFIX = f"s3://{BUCKET}/{STAGE}/{GENDER}/"

LOCAL_OUTPUT = "output/"

#### Make output directory

In [4]:
os.makedirs(LOCAL_OUTPUT, exist_ok=True)

## 1. Load Data

In [5]:
train = read_parquet("train_features.parquet")
stage1 = read_parquet("stage1_features.parquet")
stage2 = read_parquet("stage2_features.parquet")
feature_cols = read_parquet("feature_columns.parquet")['feature'].tolist()

print(f"Training data: {train.shape}")
print(f"Stage 1 grid: {stage1.shape}")
print(f"Stage 2 grid: {stage2.shape}")
print(f"Features: {len(feature_cols)}")

Training data: (3434, 100)
Stage 1 grid: (258131, 98)
Stage 2 grid: (65703, 97)
Features: 33


## 2. LightGBM Hyperparameters

In [6]:
lgb_params = {
    'max_depth': 3,
    'num_leaves': 8,
    'learning_rate': 0.05,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'min_child_samples': 10,
    'reg_alpha': 1.0,
    'reg_lambda': 1.0,
    'seed': 42,
    'verbose': -1,
}

N_ROUNDS = 500
EARLY_STOPPING = 50

print("LightGBM parameters:")
for k, v in lgb_params.items():
    print(f"  {k}: {v}")

LightGBM parameters:
  max_depth: 3
  num_leaves: 8
  learning_rate: 0.05
  subsample: 0.8
  colsample_bytree: 0.8
  min_child_samples: 10
  reg_alpha: 1.0
  reg_lambda: 1.0
  seed: 42
  verbose: -1


## 3. LOGO Cross-Validation

In [7]:
train_original = train[train['TeamA'] < train['TeamB']].copy()
folds = sorted(train['Fold'].unique())

oof_preds = []
cv_results = []

for fold in folds:
    train_mask = train['Fold'] != fold
    val_mask = (train_original['Fold'] == fold)
    
    X_train = train.loc[train_mask, feature_cols]
    y_train = train.loc[train_mask, 'Label']
    X_val = train_original.loc[val_mask, feature_cols]
    y_val = train_original.loc[val_mask, 'Label']
    
    if len(X_val) == 0:
        continue
    
    dtrain = lgb.Dataset(X_train, label=y_train)
    dval = lgb.Dataset(X_val, label=y_val, reference=dtrain)
    
    callbacks = [
        lgb.early_stopping(EARLY_STOPPING, verbose=False),
        lgb.log_evaluation(period=0)
    ]
    
    model = lgb.train(
        lgb_params,
        dtrain,
        num_boost_round=N_ROUNDS,
        valid_sets=[dval],
        feval=[lgb_brier_eval],
        callbacks=callbacks,
    )
    
    # With custom objective, predict returns raw scores — apply sigmoid
    raw_preds = model.predict(X_val)
    preds = 1.0 / (1.0 + np.exp(-raw_preds))
    brier = brier_score_loss(y_val, preds)
    
    cv_results.append({
        'Fold': fold,
        'BrierScore': brier,
        'Games': len(y_val),
        'BestRound': model.best_iteration
    })
    
    fold_oof = train_original.loc[val_mask, ['ID', 'Season', 'TeamA', 'TeamB', 'Label', 'Fold', 'IsStage1Val']].copy()
    fold_oof['Pred'] = preds
    oof_preds.append(fold_oof)
    
    print(f"  Fold {fold}: Brier={brier:.4f}, Games={len(y_val)}, BestRound={model.best_iteration}")

oof_df = pd.concat(oof_preds, ignore_index=True)
cv_df = pd.DataFrame(cv_results)

  Fold 1998: Brier=0.2079, Games=63, BestRound=326
  Fold 1999: Brier=0.1965, Games=63, BestRound=51
  Fold 2000: Brier=0.2105, Games=63, BestRound=142
  Fold 2001: Brier=0.2340, Games=63, BestRound=49
  Fold 2002: Brier=0.2168, Games=63, BestRound=339
  Fold 2003: Brier=0.2252, Games=63, BestRound=91
  Fold 2004: Brier=0.2124, Games=63, BestRound=89
  Fold 2005: Brier=0.2107, Games=63, BestRound=52
  Fold 2006: Brier=0.2087, Games=63, BestRound=69
  Fold 2007: Brier=0.2397, Games=63, BestRound=28
  Fold 2008: Brier=0.2227, Games=63, BestRound=298
  Fold 2009: Brier=0.2043, Games=63, BestRound=46
  Fold 2010: Brier=0.1878, Games=63, BestRound=134
  Fold 2011: Brier=0.2255, Games=63, BestRound=93
  Fold 2012: Brier=0.2143, Games=63, BestRound=74
  Fold 2013: Brier=0.1905, Games=63, BestRound=71
  Fold 2014: Brier=0.2163, Games=63, BestRound=149
  Fold 2015: Brier=0.1947, Games=63, BestRound=75
  Fold 2016: Brier=0.2460, Games=63, BestRound=67
  Fold 2017: Brier=0.2331, Games=63, BestRou

In [8]:
overall_brier = brier_score_loss(oof_df['Label'], oof_df['Pred'])
stage1_oof = oof_df[oof_df['IsStage1Val'] == 1]
stage1_brier = brier_score_loss(stage1_oof['Label'], stage1_oof['Pred']) if len(stage1_oof) > 0 else None

print(f"\nOverall OOF Brier Score: {overall_brier:.4f}")
print(f"Stage 1 (2022-2025) Brier Score: {stage1_brier:.4f}" if stage1_brier else "No Stage 1 data")
print(f"Mean Brier: {cv_df['BrierScore'].mean():.4f} +/- {cv_df['BrierScore'].std():.4f}")


Overall OOF Brier Score: 0.2175
Stage 1 (2022-2025) Brier Score: 0.2332
Mean Brier: 0.2173 +/- 0.0162


## 4. Train Final Model and Calibrate

In [9]:
final_rounds = int(cv_df['BestRound'].median())
print(f"Final model rounds: {final_rounds}")

dtrain_all = lgb.Dataset(train[feature_cols], label=train['Label'])
final_model = lgb.train(lgb_params, dtrain_all, num_boost_round=final_rounds)

# Calibration
calibrator = IsotonicRegression(out_of_bounds='clip')
calibrator.fit(oof_df['Pred'].values, oof_df['Label'].values)

oof_df['PredCalibrated'] = calibrator.predict(oof_df['Pred'].values)
calibrated_brier = brier_score_loss(oof_df['Label'], oof_df['PredCalibrated'])
print(f"OOF Brier (raw): {overall_brier:.4f}")
print(f"OOF Brier (calibrated): {calibrated_brier:.4f}")

Final model rounds: 69
OOF Brier (raw): 0.2175
OOF Brier (calibrated): 0.1359


## 5. Generate Predictions

In [10]:
# With custom objective, predict returns raw scores — apply sigmoid
raw_s1 = final_model.predict(stage1[feature_cols])
stage1['Pred_lightgbm'] = 1.0 / (1.0 + np.exp(-raw_s1))
stage1['Pred_lightgbm_calibrated'] = calibrator.predict(stage1['Pred_lightgbm'].values).clip(0.02, 0.98)

raw_s2 = final_model.predict(stage2[feature_cols])
stage2['Pred_lightgbm'] = 1.0 / (1.0 + np.exp(-raw_s2))
stage2['Pred_lightgbm_calibrated'] = calibrator.predict(stage2['Pred_lightgbm'].values).clip(0.02, 0.98)

print(f"Stage 1 predictions range: [{stage1['Pred_lightgbm_calibrated'].min():.3f}, {stage1['Pred_lightgbm_calibrated'].max():.3f}]")
print(f"Stage 2 predictions range: [{stage2['Pred_lightgbm_calibrated'].min():.3f}, {stage2['Pred_lightgbm_calibrated'].max():.3f}]")

# Evaluate on actual Stage 1 games
stage1_actual = stage1.dropna(subset=['Label'])
if len(stage1_actual) > 0:
    s1_brier = brier_score_loss(stage1_actual['Label'], stage1_actual['Pred_lightgbm_calibrated'])
    print(f"Stage 1 Brier (calibrated): {s1_brier:.4f}")

Stage 1 predictions range: [0.020, 0.980]
Stage 2 predictions range: [0.091, 0.899]
Stage 1 Brier (calibrated): 0.1235


## 6. Feature Importance

In [11]:
imp_df = pd.DataFrame({
    'Feature': feature_cols,
    'Gain': final_model.feature_importance(importance_type='gain')
}).sort_values('Gain', ascending=False)

print("Feature Importance (gain):")
for _, row in imp_df.iterrows():
    print(f"  {row['Feature']:30s}: {row['Gain']:.1f}")

Feature Importance (gain):
  SeedDiff                      : 2425.0
  EloDiff                       : 862.5
  SyntheticRankDiff             : 207.6
  SOSDiff                       : 143.3
  SeedA                         : 128.3
  SeedB                         : 125.2
  AvgPointDiffDiff              : 89.8
  IsPowerConfDiff               : 26.3
  RecentAvgPointDiffDiff        : 21.2
  WinPctDiff                    : 21.2
  RecentWinPctDiff              : 20.0
  AvgBlkDiff                    : 13.8
  AvgAstDiff                    : 11.4
  FGPctDiff                     : 10.9
  AvgStlDiff                    : 10.1
  FG3PctDiff                    : 9.1
  ScoreStdDiff                  : 9.0
  DefEffDiff                    : 8.4
  OppFG3PctDiff                 : 6.1
  RecentOffEffDiff              : 6.0
  AvgTODiff                     : 5.3
  NetEffDiff                    : 3.1
  AvgPossDiff                   : 1.9
  RecentNetEffDiff              : 1.8
  RecentFGPctDiff               : 1.7
 

## 7. Save Outputs

In [12]:
outputs = {
    'oof_predictions': oof_df[['ID', 'Season', 'TeamA', 'TeamB', 'Label', 'Fold', 'IsStage1Val', 'Pred', 'PredCalibrated']],
    'stage1_predictions': stage1[['ID', 'Season', 'TeamA', 'TeamB', 'Label', 'Pred_lightgbm', 'Pred_lightgbm_calibrated']],
    'stage2_predictions': stage2[['ID', 'Season', 'TeamA', 'TeamB', 'Pred_lightgbm', 'Pred_lightgbm_calibrated']],
    'cv_results': cv_df,
}

for name, df in outputs.items():
    try:
        s3_path = f"{OUTPUT_PREFIX}{name}.parquet"
        df.to_parquet(s3_path, index=False)
        print(f"Saved to S3: {s3_path} ({df.shape})")
    except Exception as e:
        print(f"S3 save failed for {name}: {e}")

Saved to S3: s3://march-machine-learning-mania-2026/05_models/lightgbm/womens/oof_predictions.parquet ((1717, 9))
Saved to S3: s3://march-machine-learning-mania-2026/05_models/lightgbm/womens/stage1_predictions.parquet ((258131, 7))
Saved to S3: s3://march-machine-learning-mania-2026/05_models/lightgbm/womens/stage2_predictions.parquet ((65703, 6))
Saved to S3: s3://march-machine-learning-mania-2026/05_models/lightgbm/womens/cv_results.parquet ((27, 4))


## 8. Summary

In [13]:
print("=" * 60)
print("LIGHTGBM MODEL SUMMARY — WOMEN'S")
print("=" * 60)
print(f"\nOOF Brier Score (raw): {overall_brier:.4f}")
print(f"OOF Brier Score (calibrated): {calibrated_brier:.4f}")
print(f"CV Mean Brier: {cv_df['BrierScore'].mean():.4f} +/- {cv_df['BrierScore'].std():.4f}")
print(f"Final model rounds: {final_rounds}")

LIGHTGBM MODEL SUMMARY — WOMEN'S

OOF Brier Score (raw): 0.2175
OOF Brier Score (calibrated): 0.1359
CV Mean Brier: 0.2173 +/- 0.0162
Final model rounds: 69
